# Step 3C - Dev Matched KL Sweep

This notebook stays on `dev_tune_200` and uses the **subject gate only** for the serious contenders.

It does not touch:

```text
analysis_500
ablation_500
final_test_500
final_test_full
```

Goal:

```text
1. Reuse Step 3B subject-gated runs at guidance_scale = 1.0.
2. Add extra guidance-scale points for myopic_score_gated_subject, no_rollout_bridge_gated_subject, and mc_bridge_gated_subject.
3. Build a matched-KL and matched-rewrite comparison among the serious contenders.
4. Produce a target-length breakdown for the best feasible point in each family.
```

This is the dev-only decision point for whether the bridge still has a differentiated tradeoff under a fair shared gate.


In [ ]:
%%bash
set -euo pipefail

pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"


In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import subprocess
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "git_commit": commit,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step3c.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)
print("Wrote:", out)
print(json.dumps(env, indent=2))


## Preflight

In [ ]:
from pathlib import Path
import csv
import json
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1/report_summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject/summary.json",
]
for name in required_files:
    path = PROJECT_DIR / name
    assert path.exists(), f"Missing required file: {path}"

thresholds_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
with thresholds_path.open() as f:
    thresholds = json.load(f)
expected_selfloc = float(thresholds["selfloc_base"])
assert 0.0 < expected_selfloc < 1.0, expected_selfloc

gated_report_summary = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1/report_summary.json"
with gated_report_summary.open() as f:
    gated_report = json.load(f)
assert gated_report["primary_paraphrase_bucket"] == "declarative_paraphrases"
assert gated_report["constraints_enabled"] is True
assert gated_report["analysis_500_used"] is False

feasible_scores_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_gated_v1/selection_scores_feasible.csv"
with feasible_scores_path.open(newline="") as f:
    feasible_rows = list(csv.DictReader(f))
assert feasible_rows, f"No rows in {feasible_scores_path}"
feasible_methods = {row["method"] for row in feasible_rows}
required_feasible = {
    "prompt_memory_gated_subject",
    "myopic_score_gated_subject",
    "no_rollout_bridge_gated_subject",
    "mc_bridge_gated_subject",
}
missing = required_feasible - feasible_methods
assert not missing, f"Missing feasible subject-gated contenders: {missing}"

print("Python:", sys.version)
print("Project dir OK:", PROJECT_DIR)
print("SelfLoc base:", expected_selfloc)
print("Step 3B gated report OK")
print("Feasible subject-gated contenders:", sorted(required_feasible))


## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025",
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050",
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200",
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs025",
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs050",
    project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs200",
    project / "runs/counterfact_direction1_v1/dev_tune_200_matched_kl_sweep_subject_v1",
]
for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )
print("Overwrite guard passed.")


## Reference Points Reused From Step 3B

This notebook reuses these already-completed `guidance_scale = 1.0` subject-gated runs:

```text
prompt_memory_gated_subject
myopic_score_gated_subject
no_rollout_bridge_gated_subject
mc_bridge_gated_subject
```

So the only new decoding here is for additional guidance scales.


## Step 3C.1: Subject-Gated Bridge Controls, guidance_scale = 0.25

Runs:

```text
myopic_score_gated_subject,no_rollout_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025   --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 0.25   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025/stdout.log


## Step 3C.2: Subject-Gated Bridge Controls, guidance_scale = 0.5

Runs:

```text
myopic_score_gated_subject,no_rollout_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050   --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 0.5   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050/stdout.log


## Step 3C.3: Subject-Gated Bridge Controls, guidance_scale = 2.0

Runs:

```text
myopic_score_gated_subject,no_rollout_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200   --methods myopic_score_gated_subject,no_rollout_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 2.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200/stdout.log


## Step 3C.4: Subject-Gated MC Bridge, guidance_scale = 0.25

Runs:

```text
mc_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs025

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs025   --methods mc_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 0.25   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs025/stdout.log


## Step 3C.5: Subject-Gated MC Bridge, guidance_scale = 0.5

Runs:

```text
mc_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs050

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs050   --methods mc_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 0.5   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs050/stdout.log


## Step 3C.6: Subject-Gated MC Bridge, guidance_scale = 2.0

Runs:

```text
mc_bridge_gated_subject
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs200

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs200   --methods mc_bridge_gated_subject   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 2.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs200/stdout.log


## Validate Sweep Run Configs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
method_specific = {
    "dev_tune_200_sweep_bridge_controls_subject_gs025": {
        "methods": ["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"],
        "guidance_scale": 0.25,
        "mc_rollouts": 0,
    },
    "dev_tune_200_sweep_bridge_controls_subject_gs050": {
        "methods": ["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"],
        "guidance_scale": 0.5,
        "mc_rollouts": 0,
    },
    "dev_tune_200_sweep_bridge_controls_subject_gs200": {
        "methods": ["myopic_score_gated_subject", "no_rollout_bridge_gated_subject"],
        "guidance_scale": 2.0,
        "mc_rollouts": 0,
    },
    "dev_tune_200_sweep_mc_bridge_subject_gs025": {
        "methods": ["mc_bridge_gated_subject"],
        "guidance_scale": 0.25,
        "mc_rollouts": 2,
    },
    "dev_tune_200_sweep_mc_bridge_subject_gs050": {
        "methods": ["mc_bridge_gated_subject"],
        "guidance_scale": 0.5,
        "mc_rollouts": 2,
    },
    "dev_tune_200_sweep_mc_bridge_subject_gs200": {
        "methods": ["mc_bridge_gated_subject"],
        "guidance_scale": 2.0,
        "mc_rollouts": 2,
    },
}

expected_protocol = {
    "protocol_version": "counterfact_direction1_v1",
    "edit_access": "given_at_edit_time",
    "training_access": "none",
    "hyperparameter_access": "dev_tune_only",
    "split_role": "dev_tune_200",
    "seed": 0,
    "model_id": "GSAI-ML/LLaDA-8B-Base",
    "eval_samples": 4,
}

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

for run_name, expected in method_specific.items():
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    cfg_path = run_dir / "run_config.json"
    rows_path = run_dir / "per_case_results.jsonl"
    summary_path = run_dir / "summary.json"
    assert cfg_path.exists(), f"Missing config: {cfg_path}"
    assert rows_path.exists(), f"Missing rows: {rows_path}"
    assert summary_path.exists(), f"Missing summary: {summary_path}"

    with cfg_path.open() as f:
        cfg = json.load(f)
    for key, value in expected_protocol.items():
        assert cfg.get(key) == value, f"{cfg_path}: expected {key}={value!r}, got {cfg.get(key)!r}"
    assert cfg["methods"] == expected["methods"], f"{cfg_path}: expected methods {expected['methods']}, got {cfg['methods']}"
    rollout = cfg.get("rollout_config", {})
    assert rollout.get("gate_mode") == "subject"
    assert rollout.get("guidance_scale") == expected["guidance_scale"]
    assert rollout.get("mc_rollouts") == expected["mc_rollouts"]
    assert rollout.get("target_logit_bias") == 0.0

    rows = list(read_jsonl(rows_path))
    assert rows, f"No rows in {rows_path}"
    row_methods = {row.get("method") for row in rows}
    assert row_methods == set(expected["methods"]), f"{run_name}: expected {expected['methods']}, got {row_methods}"

    by_method_bucket = defaultdict(set)
    for row in rows:
        method = row.get("method")
        bucket = row.get("bucket")
        edit_id = str(row.get("edit_id") or row.get("case_id"))
        if method in expected["methods"] and bucket in {"rewrite", "declarative_paraphrases", "near_locality", "far_locality"}:
            by_method_bucket[(method, bucket)].add(edit_id)
        if method in expected["methods"]:
            assert row.get("gate_mode") == "subject"
            assert row.get("method_variant") == method
            assert row.get("gate_activation_rate") is not None

    for method in expected["methods"]:
        for bucket in ["rewrite", "declarative_paraphrases", "near_locality", "far_locality"]:
            n = len(by_method_bucket[(method, bucket)])
            assert n == 200, f"{run_name}/{method}/{bucket}: expected 200 edits, got {n}"

    print("Config and completeness OK:", run_name)

print("All Step 3C sweep configs and row completeness checks OK.")


## Build Step 3C Subject-Gated Sweep Report

In [ ]:
import csv
import json
import math
import sys
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_matched_kl_sweep_subject_v1"
out_dir.mkdir(parents=True, exist_ok=True)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from llada_experiment_reports import paired_bootstrap_delta_by_case

thresholds_path = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
with thresholds_path.open() as f:
    thresholds = json.load(f)
selfloc_base = float(thresholds["selfloc_base"])
near_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["near_locality"])
far_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["far_locality"])
malformed_budget = float(thresholds["malformed_span_rate_budget"])
gpu_budget = float(thresholds["gpu_minutes_per_edit_budget"])

run_specs = [
    {
        "label": "prompt_memory_gated_subject_gs1.0",
        "family": "prompt_memory",
        "guidance_scale": 1.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_gated_prompt_memory_subject",
        "method": "prompt_memory_gated_subject",
    },
    {
        "label": "myopic_score_gated_subject_gs1.0",
        "family": "myopic_score",
        "guidance_scale": 1.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject",
        "method": "myopic_score_gated_subject",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs1.0",
        "family": "no_rollout_bridge",
        "guidance_scale": 1.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_gated_bridge_controls_subject",
        "method": "no_rollout_bridge_gated_subject",
    },
    {
        "label": "mc_bridge_gated_subject_gs1.0",
        "family": "mc_bridge",
        "guidance_scale": 1.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_gated_mc_bridge_subject",
        "method": "mc_bridge_gated_subject",
    },
    {
        "label": "myopic_score_gated_subject_gs0.25",
        "family": "myopic_score",
        "guidance_scale": 0.25,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025",
        "method": "myopic_score_gated_subject",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs0.25",
        "family": "no_rollout_bridge",
        "guidance_scale": 0.25,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs025",
        "method": "no_rollout_bridge_gated_subject",
    },
    {
        "label": "mc_bridge_gated_subject_gs0.25",
        "family": "mc_bridge",
        "guidance_scale": 0.25,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs025",
        "method": "mc_bridge_gated_subject",
    },
    {
        "label": "myopic_score_gated_subject_gs0.5",
        "family": "myopic_score",
        "guidance_scale": 0.5,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050",
        "method": "myopic_score_gated_subject",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs0.5",
        "family": "no_rollout_bridge",
        "guidance_scale": 0.5,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs050",
        "method": "no_rollout_bridge_gated_subject",
    },
    {
        "label": "mc_bridge_gated_subject_gs0.5",
        "family": "mc_bridge",
        "guidance_scale": 0.5,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs050",
        "method": "mc_bridge_gated_subject",
    },
    {
        "label": "myopic_score_gated_subject_gs2.0",
        "family": "myopic_score",
        "guidance_scale": 2.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200",
        "method": "myopic_score_gated_subject",
    },
    {
        "label": "no_rollout_bridge_gated_subject_gs2.0",
        "family": "no_rollout_bridge",
        "guidance_scale": 2.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_bridge_controls_subject_gs200",
        "method": "no_rollout_bridge_gated_subject",
    },
    {
        "label": "mc_bridge_gated_subject_gs2.0",
        "family": "mc_bridge",
        "guidance_scale": 2.0,
        "run_dir": project / "runs/counterfact_direction1_v1/dev_tune_200_sweep_mc_bridge_subject_gs200",
        "method": "mc_bridge_gated_subject",
    },
]

spec_by_label = {spec['label']: spec for spec in run_specs}

def load_json(path):
    with path.open() as f:
        return json.load(f)


def read_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def mean_or_none(values):
    values = [float(v) for v in values if v is not None]
    return sum(values) / len(values) if values else None


def harmonic_mean(values, eps=1e-12):
    vals = [max(float(v), eps) for v in values]
    if not vals:
        return 0.0
    return len(vals) / sum(1.0 / v for v in vals)


def aggregate_method_rows(rows, method):
    filtered = [r for r in rows if r.get('method_variant') == method]
    assert filtered, f'Missing method rows for {method}'
    grouped = defaultdict(list)
    for row in filtered:
        grouped[(row['bucket'], str(row.get('edit_id') or row.get('case_id')))].append(row)
    bucket_metrics = defaultdict(lambda: defaultdict(list))
    target_length_metrics = defaultdict(lambda: defaultdict(list))
    for (bucket, edit_id), items in grouped.items():
        sample = items[0]
        metrics = {
            'exact_rate': mean_or_none(item.get('exact_rate') for item in items),
            'token_f1': mean_or_none(item.get('token_f1') for item in items),
            'malformed_rate': mean_or_none(item.get('malformed_rate') for item in items),
            'target_false_positive_rate': mean_or_none(item.get('target_false_positive_rate') for item in items),
            'sparse_guidance_kl': mean_or_none(item.get('sparse_guidance_kl') for item in items),
            'gate_activation_rate': mean_or_none(item.get('gate_activation_rate') for item in items),
        }
        for key, value in metrics.items():
            if value is not None:
                bucket_metrics[bucket][key].append(value)
                target_length_metrics[(str(sample.get('target_length_bin')), bucket)][key].append(value)

    bucket_summary = {}
    for bucket, metric_lists in bucket_metrics.items():
        bucket_summary[bucket] = {key: mean_or_none(values) for key, values in metric_lists.items()}
        bucket_summary[bucket]['num_edits'] = len({str(r.get('edit_id') or r.get('case_id')) for r in filtered if r.get('bucket') == bucket})

    target_length_summary = []
    for (length_bin, bucket), metric_lists in sorted(target_length_metrics.items()):
        row = {
            'target_length_bin': length_bin,
            'bucket': bucket,
            'num_edits': len({str(r.get('edit_id') or r.get('case_id')) for r in filtered if str(r.get('target_length_bin')) == length_bin and r.get('bucket') == bucket}),
        }
        for key, values in metric_lists.items():
            row[f'mean_{key}'] = mean_or_none(values)
        target_length_summary.append(row)

    return bucket_summary, target_length_summary


def rows_for_label(label):
    spec = spec_by_label[label]
    rows = run_cache[label]['rows']
    return [r for r in rows if r.get('method_variant') == spec['method']]


def alias_rows(label):
    return [{**row, 'method_variant': label} for row in rows_for_label(label)]


sweep_rows = []
target_length_rows = []
constraint_rows = []
run_cache = {}
for spec in run_specs:
    summary = load_json(spec['run_dir'] / 'summary.json')
    rows = list(read_jsonl(spec['run_dir'] / 'per_case_results.jsonl'))
    run_cache[spec['label']] = {'summary': summary, 'rows': rows}
    bucket_summary, target_length_summary = aggregate_method_rows(rows, spec['method'])
    for required_bucket in ['rewrite', 'declarative_paraphrases', 'near_locality', 'far_locality']:
        assert bucket_summary[required_bucket]['num_edits'] == 200, (spec['label'], required_bucket, bucket_summary[required_bucket]['num_edits'])

    rewrite_exact = bucket_summary['rewrite'].get('exact_rate')
    paraphrase_exact = bucket_summary['declarative_paraphrases'].get('exact_rate')
    locality_exact = mean_or_none([
        bucket_summary['near_locality'].get('exact_rate'),
        bucket_summary['far_locality'].get('exact_rate'),
    ])
    clipped_self_norm = min(locality_exact / selfloc_base, 1.0) if locality_exact is not None else None
    selection_score = harmonic_mean([rewrite_exact, paraphrase_exact, clipped_self_norm]) if None not in [rewrite_exact, paraphrase_exact, clipped_self_norm] else None
    near_tfpr = bucket_summary['near_locality'].get('target_false_positive_rate')
    far_tfpr = bucket_summary['far_locality'].get('target_false_positive_rate')
    max_malformed = max(
        bucket_summary[bucket].get('malformed_rate') or 0.0
        for bucket in bucket_summary
    )
    gate_activation_rewrite = bucket_summary['rewrite'].get('gate_activation_rate')
    gate_activation_decl = bucket_summary['declarative_paraphrases'].get('gate_activation_rate')
    primary_kl = mean_or_none([
        bucket_summary['rewrite'].get('sparse_guidance_kl'),
        bucket_summary['declarative_paraphrases'].get('sparse_guidance_kl'),
    ])
    efficiency = summary.get('efficiency', {})
    wall_time_seconds = float(efficiency.get('wall_time_seconds') or 0.0)
    method_count = len({r.get('method_variant') for r in rows if r.get('method_variant')}) or 1
    gpu_minutes_per_edit = wall_time_seconds / 60.0 / 200.0 / method_count if wall_time_seconds else None

    violations = []
    if near_tfpr is not None and near_tfpr > near_tfpr_budget:
        violations.append(f'near_tfpr>{near_tfpr_budget:.6g}')
    if far_tfpr is not None and far_tfpr > far_tfpr_budget:
        violations.append(f'far_tfpr>{far_tfpr_budget:.6g}')
    if max_malformed is not None and max_malformed > malformed_budget:
        violations.append(f'malformed>{malformed_budget:.6g}')
    if gpu_minutes_per_edit is None:
        violations.append('gpu_minutes_per_edit_missing')
    elif gpu_minutes_per_edit > gpu_budget:
        violations.append(f'gpu_minutes_per_edit>{gpu_budget:.6g}')
    constraint_pass = not violations

    sweep_rows.append({
        'label': spec['label'],
        'family': spec['family'],
        'guidance_scale': spec['guidance_scale'],
        'method': spec['method'],
        'rewrite_exact': rewrite_exact,
        'declarative_paraphrases_exact': paraphrase_exact,
        'locality_exact': locality_exact,
        'clipped_self_normalized_locality': clipped_self_norm,
        'selection_score': selection_score,
        'near_locality_tfpr': near_tfpr,
        'far_locality_tfpr': far_tfpr,
        'max_malformed_rate': max_malformed,
        'gpu_minutes_per_edit': gpu_minutes_per_edit,
        'primary_sparse_guidance_kl': primary_kl,
        'rewrite_sparse_guidance_kl': bucket_summary['rewrite'].get('sparse_guidance_kl'),
        'declarative_sparse_guidance_kl': bucket_summary['declarative_paraphrases'].get('sparse_guidance_kl'),
        'gate_activation_rate_rewrite': gate_activation_rewrite,
        'gate_activation_rate_declarative_paraphrases': gate_activation_decl,
        'gate_activation_rate_near_locality': bucket_summary['near_locality'].get('gate_activation_rate'),
        'gate_activation_rate_far_locality': bucket_summary['far_locality'].get('gate_activation_rate'),
        'constraint_pass': constraint_pass,
        'constraint_violations': ';'.join(violations),
        'feasible_selection_score': selection_score if constraint_pass else None,
    })

    constraint_rows.append({
        'label': spec['label'],
        'family': spec['family'],
        'guidance_scale': spec['guidance_scale'],
        'constraint_pass': constraint_pass,
        'constraint_violations': ';'.join(violations),
        'near_locality_tfpr': near_tfpr,
        'near_locality_tfpr_budget': near_tfpr_budget,
        'far_locality_tfpr': far_tfpr,
        'far_locality_tfpr_budget': far_tfpr_budget,
        'max_malformed_rate': max_malformed,
        'malformed_budget': malformed_budget,
        'gpu_minutes_per_edit': gpu_minutes_per_edit,
        'gpu_budget': gpu_budget,
    })

    for row in target_length_summary:
        row.update({
            'label': spec['label'],
            'family': spec['family'],
            'guidance_scale': spec['guidance_scale'],
            'method': spec['method'],
        })
        target_length_rows.append(row)

feasible_rows = [row for row in sweep_rows if row['constraint_pass']]
feasible_rows.sort(key=lambda row: row['feasible_selection_score'], reverse=True)

# Best feasible point per family for target-length reporting.
best_by_family = {}
for row in feasible_rows:
    best_by_family.setdefault(row['family'], row)
selected_labels = {row['label'] for row in best_by_family.values()}
selected_target_length_rows = [row for row in target_length_rows if row['label'] in selected_labels]

# Matched-KL and matched-rewrite comparisons.
def nearest_match(source_row, candidates, key):
    source_value = source_row.get(key)
    valid = [row for row in candidates if row.get(key) is not None]
    if source_value is None or not valid:
        return None
    return min(valid, key=lambda row: abs(float(row[key]) - float(source_value)))

matched_kl_rows = []
matched_rewrite_rows = []
matched_bootstrap_rows = []
family_rows = defaultdict(list)
for row in feasible_rows:
    family_rows[row['family']].append(row)

comparison_specs = []
for mc_row in family_rows.get('mc_bridge', []):
    for compare_family in ['myopic_score', 'no_rollout_bridge']:
        match = nearest_match(mc_row, family_rows.get(compare_family, []), 'primary_sparse_guidance_kl')
        if match is not None:
            comparison = {
                'match_type': 'matched_kl',
                'anchor_family': 'mc_bridge',
                'anchor_label': mc_row['label'],
                'anchor_guidance_scale': mc_row['guidance_scale'],
                'compare_family': compare_family,
                'compare_label': match['label'],
                'compare_guidance_scale': match['guidance_scale'],
                'anchor_primary_sparse_guidance_kl': mc_row['primary_sparse_guidance_kl'],
                'compare_primary_sparse_guidance_kl': match['primary_sparse_guidance_kl'],
                'kl_gap': abs(float(mc_row['primary_sparse_guidance_kl']) - float(match['primary_sparse_guidance_kl'])),
                'rewrite_delta_anchor_minus_compare': float(mc_row['rewrite_exact']) - float(match['rewrite_exact']),
                'paraphrase_delta_anchor_minus_compare': float(mc_row['declarative_paraphrases_exact']) - float(match['declarative_paraphrases_exact']),
                'locality_delta_anchor_minus_compare': float(mc_row['locality_exact']) - float(match['locality_exact']),
                'feasible_score_delta_anchor_minus_compare': float(mc_row['feasible_selection_score']) - float(match['feasible_selection_score']),
            }
            matched_kl_rows.append(comparison)
            comparison_specs.append(comparison)
    for compare_family in ['prompt_memory', 'myopic_score', 'no_rollout_bridge']:
        match_rw = nearest_match(mc_row, family_rows.get(compare_family, []), 'rewrite_exact')
        if match_rw is not None:
            comparison = {
                'match_type': 'matched_rewrite',
                'anchor_family': 'mc_bridge',
                'anchor_label': mc_row['label'],
                'anchor_guidance_scale': mc_row['guidance_scale'],
                'compare_family': compare_family,
                'compare_label': match_rw['label'],
                'compare_guidance_scale': match_rw['guidance_scale'],
                'anchor_rewrite_exact': mc_row['rewrite_exact'],
                'compare_rewrite_exact': match_rw['rewrite_exact'],
                'rewrite_gap': abs(float(mc_row['rewrite_exact']) - float(match_rw['rewrite_exact'])),
                'paraphrase_delta_anchor_minus_compare': float(mc_row['declarative_paraphrases_exact']) - float(match_rw['declarative_paraphrases_exact']),
                'locality_delta_anchor_minus_compare': float(mc_row['locality_exact']) - float(match_rw['locality_exact']),
                'feasible_score_delta_anchor_minus_compare': float(mc_row['feasible_selection_score']) - float(match_rw['feasible_selection_score']),
            }
            matched_rewrite_rows.append(comparison)
            comparison_specs.append(comparison)

metrics_to_bootstrap = [
    ('rewrite', 'exact_rate'),
    ('declarative_paraphrases', 'exact_rate'),
    ('near_locality', 'exact_rate'),
    ('far_locality', 'exact_rate'),
]

for comparison in comparison_specs:
    anchor_rows = alias_rows(comparison['anchor_label'])
    compare_rows = alias_rows(comparison['compare_label'])
    pair_rows = anchor_rows + compare_rows
    for bucket, metric in metrics_to_bootstrap:
        boot = paired_bootstrap_delta_by_case(
            pair_rows,
            candidate_method=comparison['anchor_label'],
            baseline_method=comparison['compare_label'],
            bucket=bucket,
            metric=metric,
            samples=1000,
            seed=0,
        )
        matched_bootstrap_rows.append({
            'match_type': comparison['match_type'],
            'anchor_family': comparison['anchor_family'],
            'anchor_label': comparison['anchor_label'],
            'anchor_guidance_scale': comparison['anchor_guidance_scale'],
            'compare_family': comparison['compare_family'],
            'compare_label': comparison['compare_label'],
            'compare_guidance_scale': comparison['compare_guidance_scale'],
            'bucket': bucket,
            'metric': metric,
            'mean_delta_anchor_minus_compare': boot['mean_delta'],
            'ci_lower': boot['ci_low'],
            'ci_upper': boot['ci_high'],
            'num_edits': boot.get('num_edits') or boot.get('num_cases'),
        })

# Write CSV helpers.
def write_csv(path, rows):
    if not rows:
        path.write_text('')
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

write_csv(out_dir / 'sweep_summary.csv', sweep_rows)
write_csv(out_dir / 'feasible_ranking.csv', feasible_rows)
write_csv(out_dir / 'constraint_summary.csv', constraint_rows)
write_csv(out_dir / 'matched_kl_comparisons.csv', matched_kl_rows)
write_csv(out_dir / 'matched_rewrite_comparisons.csv', matched_rewrite_rows)
write_csv(out_dir / 'matched_bootstrap.csv', matched_bootstrap_rows)
write_csv(out_dir / 'target_length_breakdown.csv', selected_target_length_rows)

# Plots.
family_colors = {
    'prompt_memory': '#1f77b4',
    'myopic_score': '#ff7f0e',
    'no_rollout_bridge': '#2ca02c',
    'mc_bridge': '#d62728',
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for family, rows in family_rows.items():
    xs = [row['primary_sparse_guidance_kl'] for row in rows]
    ys = [row['rewrite_exact'] for row in rows]
    axes[0].plot(xs, ys, marker='o', label=family, color=family_colors.get(family))
    for row in rows:
        axes[0].annotate(str(row['guidance_scale']), (row['primary_sparse_guidance_kl'], row['rewrite_exact']), fontsize=8)
axes[0].set_xlabel('Primary sparse guidance KL')
axes[0].set_ylabel('Rewrite exact')
axes[0].set_title('Matched-KL Tradeoff')
axes[0].legend()
for family, rows in family_rows.items():
    xs = [row['rewrite_exact'] for row in rows]
    ys = [row['declarative_paraphrases_exact'] for row in rows]
    axes[1].plot(xs, ys, marker='o', label=family, color=family_colors.get(family))
    for row in rows:
        axes[1].annotate(str(row['guidance_scale']), (row['rewrite_exact'], row['declarative_paraphrases_exact']), fontsize=8)
axes[1].set_xlabel('Rewrite exact')
axes[1].set_ylabel('Declarative paraphrases exact')
axes[1].set_title('Matched-Rewrite Comparison')
axes[1].legend()
fig.tight_layout()
fig.savefig(out_dir / 'matched_tradeoff_plots.png', dpi=200)
plt.close(fig)

summary = {
    'protocol_version': 'counterfact_direction1_v1',
    'split_role': 'dev_tune_200',
    'gate_mode': 'subject',
    'selfloc_base': selfloc_base,
    'families': sorted({row['family'] for row in sweep_rows}),
    'guidance_scales': sorted({row['guidance_scale'] for row in sweep_rows}),
    'selected_best_feasible_by_family': best_by_family,
    'artifacts': {
        'sweep_summary': str(out_dir / 'sweep_summary.csv'),
        'feasible_ranking': str(out_dir / 'feasible_ranking.csv'),
        'constraint_summary': str(out_dir / 'constraint_summary.csv'),
        'matched_kl_comparisons': str(out_dir / 'matched_kl_comparisons.csv'),
        'matched_rewrite_comparisons': str(out_dir / 'matched_rewrite_comparisons.csv'),
        'matched_bootstrap': str(out_dir / 'matched_bootstrap.csv'),
        'target_length_breakdown': str(out_dir / 'target_length_breakdown.csv'),
        'matched_tradeoff_plots': str(out_dir / 'matched_tradeoff_plots.png'),
    },
}
with (out_dir / 'report_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Wrote:', out_dir)
print('Feasible ranking:')
for row in feasible_rows:
    print(row)


## Expected Outcome

This notebook should answer the bridge-specific question that Step 3B left unresolved:

```text
At matched sparse guidance KL, does mc_bridge_gated_subject beat myopic_score_gated_subject?
At matched rewrite level, does mc_bridge_gated_subject have better paraphrase or locality behavior?
```

If not, the honest next conclusion is that rollout-aware bridge control is viable and beats no-rollout bridge, but simple gated baselines still dominate CounterFact under the current protocol.
